## Show, Attend and Tell — Training (Flickr8k)


Clone repository

In [1]:
!git clone https://github.com/seanzhangw/show-attend-tell.git

Cloning into 'show-attend-tell'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 67 (delta 20), reused 57 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (67/67), 37.59 KiB | 6.26 MiB/s, done.
Resolving deltas: 100% (20/20), done.


Pull recent changes

In [2]:
%cd /content/show-attend-tell
!git pull

/content/show-attend-tell
Already up to date.


Get dataset

In [3]:
%cd /content/show-attend-tell/data
!python get_flickr8k.py

/content/show-attend-tell/data
100% 1.04G/1.04G [00:05<00:00, 186MB/s] 
Extracting files...
Downloaded to: /root/.cache/kagglehub/datasets/adityajn105/flickr8k/versions/1
Dataset available at: /content/show-attend-tell/data/flickr8k


In [4]:
%cd /content/show-attend-tell/code

import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

import config
from datasets.flickr8k import collate_fn, build_flickr8k_dataset_split
from eval.greedy_decode import greedy_decode
from models.encoder import EncoderCNN
from models.decoder import Decoder


/content/show-attend-tell/code


Define Hyperparameters

In [6]:
# Hyperparameters (edit as needed)
epochs = 4
batch_size = config.BATCH_SIZE
lr = 1e-4
embed_dim = 512
hidden_dim = 512
attention_dim = 512

dropout = 0.5
lambda_att = 1.0
grad_clip = 5.0
freeze_encoder = True
val_ratio = 0.1
split_seed = 42
# BLEU-4: cap images per eval for speed (None = all val images)
bleu_max_images = 500

Build datasets, dataloaders

In [7]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            # ImageNet normalization for pre-trained ResNet (ImageNet dataset mean and std)
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

# Dataset: image-level train/val split; vocab from train captions only
(
    train_set,
    val_set,
    val_image_ids,
    full_captions_map,
    word2idx,
    idx2word,
) = build_flickr8k_dataset_split(
    config,
    transform=transform,
    val_ratio=val_ratio,
    seed=split_seed,
)
pad_idx = word2idx["<pad>"]
print(
    f"Train samples: {len(train_set)} | Val samples: {len(val_set)} | "
    f"Val images: {len(val_image_ids)} | Vocab size: {len(word2idx)}"
)

train_loader = DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_set,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
)

Using device: cuda
Train samples: 36410 | Val samples: 4045 | Val images: 809 | Vocab size: 8363


Initialize models, loss, and optimizer

In [8]:
# Models
encoder = EncoderCNN().to(device)
decoder = Decoder(
    vocab_size=len(word2idx),
    embed_dim=embed_dim,
    feature_dim=2048,
    hidden_dim=hidden_dim,
    attention_dim=attention_dim,
    dropout=dropout,
).to(device)

# Loss / Optimizer
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

params = list(decoder.parameters())
if not freeze_encoder:
    params += [p for p in encoder.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr=lr)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 171MB/s] 


Training

In [ ]:
from training.loop import evaluate_caption_metrics, run_training_loop

CHECKPOINT_DIR = "./checkpoints"
RESUME_PATH = None  # e.g. "./checkpoints/best_by_val_loss.pt"
EVAL_METRICS = ["bleu1", "bleu2", "bleu3", "bleu4"]  # add "meteor" after nltk.download(...)
BEST_BY = "val_loss"
BEST_MODE = "min"  # "max" when BEST_BY is a score to maximize (e.g. "bleu4")

_, best_ckpt_path, _ = run_training_loop(
    encoder,
    decoder,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs,
    val_image_ids,
    full_captions_map,
    transform,
    word2idx,
    idx2word,
    config.IMAGE_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    eval_metric_names=EVAL_METRICS,
    best_by=BEST_BY,
    best_mode=BEST_MODE,
    resume_path=RESUME_PATH,
    max_decode_len=config.MAX_LEN,
    bleu_max_images=bleu_max_images,
    lambda_att=lambda_att,
    grad_clip=grad_clip,
    print_every=100,
    freeze_encoder=freeze_encoder,
    hparams={
        "embed_dim": embed_dim,
        "hidden_dim": hidden_dim,
        "attention_dim": attention_dim,
        "dropout": dropout,
        "lambda_att": lambda_att,
        "freeze_encoder": freeze_encoder,
        "lr": lr,
        "batch_size": batch_size,
        "val_ratio": val_ratio,
        "split_seed": split_seed,
    },
)



Epoch 1/4
[Train]   100/1138 loss=3.4662 (CE=3.2955, Att=0.3801) | 2.95 it/s | ETA 5.9 min
[Train]   200/1138 loss=3.4756 (CE=3.2195, Att=0.3800) | 2.95 it/s | ETA 5.3 min
[Train]   300/1138 loss=3.4668 (CE=3.0588, Att=0.3806) | 2.96 it/s | ETA 4.7 min
[Train]   400/1138 loss=3.4664 (CE=3.0327, Att=0.3803) | 2.95 it/s | ETA 4.2 min
[Train]   500/1138 loss=3.4613 (CE=2.7881, Att=0.3803) | 2.96 it/s | ETA 3.6 min
[Train]   600/1138 loss=3.4565 (CE=2.7447, Att=0.3826) | 2.95 it/s | ETA 3.0 min
[Train]   700/1138 loss=3.4527 (CE=3.1128, Att=0.3819) | 2.95 it/s | ETA 2.5 min
[Train]   800/1138 loss=3.4500 (CE=3.2800, Att=0.3795) | 2.95 it/s | ETA 1.9 min
[Train]   900/1138 loss=3.4442 (CE=2.8327, Att=0.3810) | 2.95 it/s | ETA 1.3 min
[Train]  1000/1138 loss=3.4412 (CE=2.9628, Att=0.3803) | 2.95 it/s | ETA 0.8 min
[Train]  1100/1138 loss=3.4370 (CE=2.9446, Att=0.3800) | 2.95 it/s | ETA 0.2 min
[Epoch 1] train_loss=3.4355 | val_loss=3.5581 | bleu1=0.5902 | bleu2=0.4223 | bleu3=0.2874 | bleu4

In [20]:
# Eval-only metrics (no training step)
metric_scores = evaluate_caption_metrics(
    encoder,
    decoder,
    val_image_ids,
    full_captions_map,
    config.IMAGE_DIR,
    transform,
    word2idx,
    idx2word,
    device,
    eval_metric_names=EVAL_METRICS,
    max_decode_len=config.MAX_LEN,
    max_images=bleu_max_images,
)
print("Eval-only metrics:", metric_scores)

Eval-only metrics: {'bleu1': 0.6040058801911062, 'bleu2': 0.42774024412009715, 'bleu3': 0.2871771219809687, 'bleu4': 0.18687759380623048}


In [12]:
from PIL import Image

num_examples = min(20, len(val_image_ids))
print(f"\nQualitative eval on {num_examples} validation images (unique)")

for i, img_name in enumerate(sorted(val_image_ids)[:num_examples]):
    path = os.path.join(config.IMAGE_DIR, img_name)
    image = Image.open(path).convert("RGB")
    image_tensor = transform(image)

    pred_tokens = greedy_decode(
        encoder,
        decoder,
        image_tensor,
        word2idx,
        idx2word,
        device,
        max_len=config.MAX_LEN,
    )
    refs = full_captions_map[img_name][:5]

    print(f"\n[{i+1:02d}] Image: {img_name}")
    print("Generated:", " ".join(pred_tokens) if pred_tokens else "<empty>")
    for j, ref in enumerate(refs[:3], start=1):
        print(f"Ref {j}: {ref}")



Qualitative eval on 20 validation images (unique)

[01] Image: 101654506_8eb26cfb60.jpg
Generated: a white dog is running through the snow
Ref 1: a brown and white dog is running through the snow .
Ref 2: a dog is running in the snow
Ref 3: a dog running through snow .

[02] Image: 104136873_5b5d41be75.jpg
Generated: a man is sitting on a rock
Ref 1: people sit on the mountainside and check out the view .
Ref 2: three people are on a hilltop overlooking a green valley .
Ref 3: three people hang out on top of a big hill .

[03] Image: 1042020065_fb3d3ba5ba.jpg
Generated: a man in a blue shirt is sitting on a boat in a lake
Ref 1: a boy in a green shirt is looking down at many inflatable boats .
Ref 2: a boy in a green shirt watches kayakers .
Ref 3: a boy looks over a railing at the many boats and rafts below in the water .

[04] Image: 1048710776_bb5b0a5c7c.jpg
Generated: a group of people are sitting on a dock at the beach
Ref 1: a couple of several people sitting on a ledge overlook

Save the trained model to Google Drive

In [14]:
from google.colab import drive
import sys

drive.mount('/content/gdrive')

# NOTE: Change this path to your own Google Drive path
base_dir = "/content/gdrive/MyDrive/show-attend-tell"
sys.path.append(base_dir)

Mounted at /content/gdrive


In [16]:
import shutil
import os

# Define the source (where the model is now) and destination (your Drive)
source_path = './checkpoints/best_by_val_loss.pt'
dest_path = os.path.join(base_dir, 'best_by_val_loss_4_epochs.pt')

# Create the directory on Drive if it doesn't exist
os.makedirs(base_dir, exist_ok=True)

# Copy the file
shutil.copyfile(source_path, dest_path)

print(f"Model successfully saved to: {dest_path}")

Model successfully saved to: /content/gdrive/MyDrive/show-attend-tell/best_by_val_loss_4_epochs.pt


Loading a model into encoder_loaded, decoder_loaded variables

In [18]:
# Load encoder/decoder from a checkpoint stored on Google Drive
from training.checkpoint import load_models_from_checkpoint_path

# Path on Drive where we saved the checkpoint above
ckpt_path = os.path.join(base_dir, "best_by_val_loss.pt")
print("Loading checkpoint from:", ckpt_path)

# Re-create model objects with the same architecture as during training
encoder_loaded = EncoderCNN().to(device)
decoder_loaded = Decoder(
    vocab_size=len(word2idx),
    embed_dim=embed_dim,
    feature_dim=2048,
    hidden_dim=hidden_dim,
    attention_dim=attention_dim,
    dropout=dropout,
).to(device)

# (Optional) create an optimizer if you want to resume training
optimizer_loaded = torch.optim.Adam(list(decoder_loaded.parameters()), lr=lr)

resume_info = load_models_from_checkpoint_path(
    ckpt_path,
    encoder_loaded,
    decoder_loaded,
    optimizer=optimizer_loaded,  # set to None if you only care about inference
    map_location=device,
)

print("Start epoch after restore:", resume_info["start_epoch"])
print("Last metrics in checkpoint:", resume_info["metrics"])
print("Tracking info:", resume_info["tracking"])

Loading checkpoint from: /content/gdrive/MyDrive/show-attend-tell/best_by_val_loss.pt
Start epoch after restore: 9
Last metrics in checkpoint: {'train_loss': 3.0788595261598934, 'val_loss': 3.447047853094386}
Tracking info: {}
